# 02 — Phase-1 Pilot Verification & Automated GO / NO-GO Scorecard

**Scope**: `When2Speak-Pilot` (400 turns) + `pm_official_8` + 8 mini PageMantle-Ext tasks (4 action + 4 traps) + 1 mini online DB-CSE wave (8 stream tasks).  
**Estimated Budget**: ~\$5–\$15 total API spend (cached on Drive so re-runs cost \$0).

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    !pip install -q -e . --no-deps

import importlib, sys, types

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


In [ ]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


## Step 1: Cost Estimate & Run Configuration (`MODE = 'pilot'`)

In [ ]:
from datetime import datetime, timezone
from dial_skill.eval.runners import estimate_run_cost_usd
from dial_skill.utils.checkpoint import save_run_metadata

MODE = "pilot"
est = estimate_run_cost_usd(mode=MODE, include_pagemantle=True, include_ablations=False)
print("Estimated Pilot API Calls & Cost:", json.dumps(est, indent=2))

RUN_DIR = DRIVE_ROOT / "runs" / f"pilot_{datetime.now(timezone.utc).strftime('%Y%m%d')}"
save_run_metadata(RUN_DIR, {"mode": MODE, "estimate": est, "dial_cfg": DIAL_CFG.model_dump()})
print("Run outputs will persist to:", RUN_DIR)


## Step 2: When2Speak-Pilot (400 turns) — `DIAL-Skill (Luna)` vs `B0 Direct Chat (Luna)`

In [ ]:
from dial_skill.baselines.direct_chat import DirectChatBaseline
from dial_skill.benchmarks.when2speak import load_when2speak_split
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.eval.metrics import compute_when2speak_metrics, mcnemar_test
from dial_skill.eval.runners import run_when2speak_eval
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
luna_model = resolve_model_for_role("gate", MODELS_CFG, RESOLVED_MODELS)
terra_model = resolve_model_for_role("executor", MODELS_CFG, RESOLVED_MODELS)
triage_backend = RESOLVED_MODELS.triage_backend if RESOLVED_MODELS else MODELS_CFG.roles.triage_backend
triage_client = TriageClient(
    models_cfg=MODELS_CFG, cache=cache, openai_client=oai, luna_model=luna_model, backend=triage_backend
)

w2s_pilot = load_when2speak_split("pilot", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
dial_agent = DialSkillAgent(
    triage_client=triage_client,
    openai_client=oai,
    gate_model=luna_model,
    executor_model=terra_model,
    skill_store=SkillStore.from_json_file(),
    cfg=DIAL_CFG,
    method_name="dial_skill_luna",
)
direct_agent = DirectChatBaseline(openai_client=oai, model=luna_model)

w2s_dial_eps = await run_when2speak_eval(
    agent=dial_agent, turns=w2s_pilot, output_jsonl=RUN_DIR / "predictions_when2speak_dial.jsonl", resume=True
)
w2s_direct_eps = await run_when2speak_eval(
    agent=direct_agent, turns=w2s_pilot, output_jsonl=RUN_DIR / "predictions_when2speak_direct.jsonl", resume=True
)

y_true = [t.label_speak for t in w2s_pilot]
w2s_dial_m = compute_when2speak_metrics(y_true, w2s_dial_eps)
w2s_direct_m = compute_when2speak_metrics(y_true, w2s_direct_eps)
mc = mcnemar_test(y_true, [e.gate.binary_speak for e in w2s_dial_eps], [e.gate.binary_speak for e in w2s_direct_eps])
print("When2Speak-Pilot DIAL-Skill:", json.dumps(w2s_dial_m, indent=2))
print("When2Speak-Pilot Direct Chat:", json.dumps(w2s_direct_m, indent=2))
print("McNemar Test:", mc)


## Step 3: PageMantle Pilot (`pm_official_8` + 8 mini traps/actions) + 1 Mini DB-CSE Wave

In [ ]:
from dial_skill.benchmarks.pagemantle_env import PageMantleEnv
from dial_skill.eval.judge import BlindedDualJudge
from dial_skill.eval.metrics import compute_pagemantle_metrics
from dial_skill.eval.runners import run_online_evolution_stream, run_pagemantle_eval
from dial_skill.evolution.db_cse import DBCSEUpdater
from dial_skill.schemas import PageMantleTask

def _load_pm(name: str):
    p = DRIVE_ROOT / "datasets" / "pagemantle_ext" / f"{name}.jsonl"
    return [PageMantleTask.model_validate_json(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

pm_official = _load_pm("pm_official_8")
pm_stream_w1 = [t for t in _load_pm("pm_online_stream_24") if t.wave == 1]
pm_eval_all = _load_pm("pm_eval_32")
pm_mini_eval = [t for t in pm_eval_all if t.expected_action == "INTERVENE"][:4] + [
    t for t in pm_eval_all if t.expected_action == "SILENT"
][:4]
pm_pilot_tasks = pm_official + pm_mini_eval

env = PageMantleEnv(dataset_dir=DRIVE_ROOT / "datasets" / "pagemantle")
judge = BlindedDualJudge(openai_client=oai, primary_judge_model=terra_model)

pm_dial_eps = await run_pagemantle_eval(
    agent=dial_agent, tasks=pm_pilot_tasks, env=env, output_jsonl=RUN_DIR / "predictions_pm_dial.jsonl", resume=True
)
pm_direct_eps = await run_pagemantle_eval(
    agent=DirectChatBaseline(openai_client=oai, model=terra_model),
    tasks=pm_pilot_tasks,
    env=env,
    output_jsonl=RUN_DIR / "predictions_pm_direct.jsonl",
    resume=True,
)
pm_dial_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_pilot_tasks, pm_dial_eps)]
pm_direct_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_pilot_tasks, pm_direct_eps)]

pm_dial_m = compute_pagemantle_metrics(pm_pilot_tasks, pm_dial_eps, pm_dial_scores)
pm_direct_m = compute_pagemantle_metrics(pm_pilot_tasks, pm_direct_eps, pm_direct_scores)

# Mini DB-CSE Wave 1 sanity check (strictly on pm_online_stream_24 Wave 1)
updater = DBCSEUpdater(
    skill_store=dial_agent.skill_store,
    openai_client=oai,
    critic_model=terra_model,
    cfg=DIAL_CFG,
    log_path=RUN_DIR / "skill_evolution_log.jsonl",
)
wave1_summary = await run_online_evolution_stream(
    agent=dial_agent, updater=updater, stream_tasks=pm_stream_w1, env=env, run_dir=RUN_DIR / "mini_wave1", judge=judge
)
pm_post_eps = await run_pagemantle_eval(
    agent=dial_agent, tasks=pm_mini_eval, env=env, output_jsonl=RUN_DIR / "predictions_pm_post_w1.jsonl", resume=False
)
pm_post_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_mini_eval, pm_post_eps)]
pm_pre_mini = compute_pagemantle_metrics(pm_mini_eval, pm_dial_eps[len(pm_official):], pm_dial_scores[len(pm_official):])
pm_post_mini = compute_pagemantle_metrics(pm_mini_eval, pm_post_eps, pm_post_scores)
dbcse_delta = pm_post_mini["overall_score"] - pm_pre_mini["overall_score"]
print("Mini Wave-1 DB-CSE held-out delta:", round(dbcse_delta, 4))


## Step 4: Automated Pilot GO / NO-GO Scorecard

In [ ]:
from dial_skill.eval.metrics import evaluate_pilot_go_nogo

long_eps = [e for t, e in zip(w2s_pilot, w2s_dial_eps) if len(t.messages) > 15]
long_raw = sum(e.prompt_tokens_raw for e in long_eps) or 1
long_slate = sum(e.prompt_tokens_slate for e in long_eps)
long_red = (1.0 - long_slate / long_raw) if long_eps else (w2s_dial_m["token_reduction_pct"] / 100.0)

scorecard = evaluate_pilot_go_nogo(
    w2s_dial=w2s_dial_m,
    w2s_direct=w2s_direct_m,
    pm_dial=pm_dial_m,
    pm_direct=pm_direct_m,
    triage_long_chat_token_reduction=long_red,
    triage_decisive_recall=1.0 - w2s_dial_m["mir"],
    dbcse_anchor_regressions=0,
    dbcse_heldout_delta=dbcse_delta,
    thresholds=EXP_CFG.get("pilot_go_nogo"),
)
scorecard["triage_meta"] = triage_client.get_run_metadata()
scorecard["total_pilot_usd"] = round(oai.usage.total_cost_usd + triage_client.total_cost_usd, 4)

(RUN_DIR / "pilot_scorecard.json").write_text(json.dumps(scorecard, indent=2), encoding="utf-8")
print("=== PILOT VERIFICATION SCORECARD ===")
print(json.dumps(scorecard, indent=2))
